In [2]:
import json
import os
import dotenv

import numpy as np
import pandas as pd

from sklearn.cluster import HDBSCAN

import sqlalchemy as sa
from sqlalchemy import select, create_engine

from models import Wards, Matches

dotenv.load_dotenv()

True

In [3]:
db = create_engine(os.environ['DB_URL'])

In [4]:
df = pd.DataFrame(pd.read_sql(
    sql=select(Wards, Matches).join(Matches).where(sa.and_(Wards.is_obs, Matches.league_id == 15728)).order_by(Matches.id.desc()),
    con=db
))

In [5]:
df['duration'] = df[['duration', 'duration_1', 'time_placed']].apply((lambda row: min(row['duration'], row['duration_1'] - row['time_placed'])), axis=1)

In [6]:
hdbscan = HDBSCAN(
    cluster_selection_epsilon=70,
    min_samples=5,
    min_cluster_size=5,
    cluster_selection_method='eom',
    n_jobs=-1,
)
hdbscan.fit(df[['x_pos', 'y_pos']])

df['cluster_id'] = hdbscan.labels_

In [7]:
df.groupby('cluster_id').size().sort_values(ascending=False)

cluster_id
-1      618
 67     235
 33     211
 4      151
 3      150
       ... 
 105      5
 21       5
 100      5
 90       5
 97       5
Length: 175, dtype: int64

In [8]:
df['advantage'] = df[['is_radiant', 'radiant_networth', 'dire_networth']].apply(lambda row: (row['radiant_networth'] - row['dire_networth']) if row['is_radiant'] else (row['dire_networth'] - row['radiant_networth']), axis=1)
df

,id,match_id,is_obs,is_radiant,is_destroyed,is_post_game,player_placed_id,player_destroyed_id,time_placed,duration,...,radiant_networth,dire_networth,id_1,league_id,duration_1,region,date,game_version,cluster_id,advantage
0,837903,7406531302,True,True,True,False,91730177.0,106305042.0,-63,34,...,700,700,7406531302,15728,2525,1,1698621223,53,167,0
1,837908,7406531302,True,False,False,False,321580662.0,NaN,-67,360,...,700,700,7406531302,15728,2525,1,1698621223,53,148,0
2,837909,7406531302,True,True,False,False,221666230.0,NaN,-60,360,...,700,700,7406531302,15728,2525,1,1698621223,53,148,0
3,837910,7406531302,True,False,False,False,256156323.0,NaN,-56,360,...,700,700,7406531302,15728,2525,1,1698621223,53,143,0
4,837919,7406531302,True,True,False,False,16497807.0,NaN,291,360,...,1553,2524,7406531302,15728,2525,1,1698621223,53,53,-971
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5370,827120,7378530387,True,False,False,False,136829091.0,NaN,-60,360,...,700,650,7378530387,15728,1405,1,1697130044,53,156,-50
5371,827121,7378530387,True,False,False,False,346412363.0,NaN,-32,360,...,700,600,7378530387,15728,1405,1,1697130044,53,88,-100
5372,827133,7378530387,True,True,False,False,10366616.0,NaN,354,360,...,2619,1071,7378530387,15728,1405,1,1697130044,53,145,1548
5373,827134,7378530387,True,False,False,False,136829091.0,NaN,357,360,...,2624,1076,7378530387,15728,1405,1,1697130044,53,52,-1548


In [9]:
grouped = df.groupby(['cluster_id', 'is_radiant']).agg({
        'x_pos': lambda x: round(x.mean()),
        'y_pos': lambda x: round(x.mean()),
        'z_pos': lambda x: x.mode().iloc[0], #- 16384,
        'duration': lambda x: round(x.mean()),
        'time_placed': lambda x: round(x.mean()),
        'advantage': lambda x: round(x.mean()),
        'is_destroyed': 'sum'
    })
grouped['amount'] = df.groupby(['cluster_id', 'is_radiant']).size()
grouped

x_pos  y_pos  z_pos  duration  time_placed  advantage  \
cluster_id is_radiant                                                          
-1         False       15528  16129  16512       275         1229        810   
           True        16887  16604  16512       261         1239       -399   
 0         False       14841  23284  16784       285         1697       -801   
           True        14827  23290  16784       193         1522       1528   
 1         False        8450  18193  16920       289         1661         77   
...                      ...    ...    ...       ...          ...        ...   
 171       True        17224  13201  16640       358         1346      -7045   
 172       False       17776  12832  16640       281         1104        138   
           True        17806  12824  16640       309          934       1148   
 173       False       17522  12932  16640       121          275        694   
           True        17509  12928  16640       270          931      -1958   

                       is_destroyed  amount  
cluster_id is_radiant                        
-1         False                112     322  
           True                 114     296  
 0         False                 15      44  
           True                  30      48  
 1         False                  8      27  
...                             ...     ...  
 171       True                   1       6  
 172       False                  4      11  
           True                   1       6  
 173       False                  2       2  
           True                   2       6  

[332 rows x 8 columns]

In [10]:
players = df[df['cluster_id'] != -1].groupby(['cluster_id', 'player_placed_id']).agg({'duration': lambda x: round(x.mean())})
players['amount'] = df.groupby(['cluster_id', 'player_placed_id']).size()
players

duration  amount
cluster_id player_placed_id                  
0          10366616.0              36       1
           77490514.0             298       2
           81306398.0             202       4
           81475303.0             266      11
           85696663.0             360       2
...                               ...     ...
173        150961567.0            139       1
           193815691.0            360       1
           196878136.0            103       1
           317880638.0            162       1
           431770905.0            360       1

[2496 rows x 2 columns]

In [11]:
players_destroyed = df[df['cluster_id'] != -1].groupby(['cluster_id', 'player_destroyed_id']).agg({'duration': lambda x: round(x.mean())})
players_destroyed['amount'] = df.groupby(['cluster_id', 'player_destroyed_id']).size()
players_destroyed

duration  amount
cluster_id player_destroyed_id                  
0          10366616.0                 42       1
           16497807.0                 37       1
           25907144.0                105       2
           77490514.0                106       3
           81475303.0                108       2
...                                  ...     ...
172        161839895.0                56       1
           317880638.0               193       1
173        134556694.0               162       1
           158847773.0               121       2
           201358612.0                20       1

[1299 rows x 2 columns]

In [12]:
for i in range(len(set(hdbscan.labels_)) - 1):
    current = grouped.loc[i]

    d = {
        'cluster_id': i,
        'x_pos': round(current['x_pos'].mean()),
        'y_pos': round(current['y_pos'].mean()),
        'z_pos': round(current['z_pos'].mean()),
        'radiant': {
            'amount': current.loc[True]['amount'],
            'destroyed': current.loc[True]['is_destroyed'],
            'advantage': current.loc[True]['advantage'],
            'duration': current.loc[True]['duration'],
            'time_placed': current.loc[True]['time_placed']
        } if True in current.index else None,
        'dire': {
            'amount': current.loc[False]['amount'],
            'destroyed': current.loc[False]['is_destroyed'],
            'advantage': current.loc[False]['advantage'],
            'duration': current.loc[False]['duration'],
            'time_placed': current.loc[False]['time_placed']
        } if False in current.index else None,
        'all': {
            'amount': current['amount'].sum(),
            'destroyed': current['is_destroyed'].sum(),
            'advantage': None,
            'duration': round(current['duration'].mean()),
            'time_placed': round(current['time_placed'].mean())
        }
    }

    print(json.dumps(d, indent=4, default=int))

{
    "cluster_id": 0,
    "x_pos": 14834,
    "y_pos": 23287,
    "z_pos": 16784,
    "radiant": {
        "amount": 48,
        "destroyed": 30,
        "advantage": 1528,
        "duration": 193,
        "time_placed": 1522
    },
    "dire": {
        "amount": 44,
        "destroyed": 15,
        "advantage": -801,
        "duration": 285,
        "time_placed": 1697
    },
    "all": {
        "amount": 92,
        "destroyed": 45,
        "advantage": null,
        "duration": 239,
        "time_placed": 1610
    }
}
{
    "cluster_id": 1,
    "x_pos": 8453,
    "y_pos": 18196,
    "z_pos": 16920,
    "radiant": {
        "amount": 23,
        "destroyed": 8,
        "advantage": -2683,
        "duration": 270,
        "time_placed": 1991
    },
    "dire": {
        "amount": 27,
        "destroyed": 8,
        "advantage": 77,
        "duration": 289,
        "time_placed": 1661
    },
    "all": {
        "amount": 50,
        "destroyed": 16,
        "advantage": null,
     

In [13]:
average_df = grouped.reset_index('is_radiant')
average_radiant = average_df[average_df['is_radiant'] == True]
average_dire = average_df[average_df['is_radiant'] == False]
average = {
    'cluster_id': -1,
    'radiant': {
        'advantage': round(average_radiant['advantage'].mean()),
        'duration': round(average_radiant['duration'].mean()),
    },
    'dire': {
        'advantage': round(average_dire['advantage'].mean()),
        'duration': round(average_dire['duration'].mean()),
    },
    'all': {
        'advantage': None,
        'duration': round(average_df['duration'].mean()),
    }
}
average

{'cluster_id': -1,
 'radiant': {'advantage': -935, 'duration': 284},
 'dire': {'advantage': 418, 'duration': 274},
 'all': {'advantage': None, 'duration': 279}}